[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/join8112/wafer-yield-portfolio/blob/main/notebooks/04_cnn_colab.ipynb)

# 04 · CNN on wafer maps (train on Colab GPU)

**Goal:** learn defect patterns directly from the map instead of handcrafted features, and check *why* the model decides with Grad-CAM.

**Model:** 3 convolutional blocks → global average pooling → linear (~0.3M parameters). Input has two channels, *is die* and *is failing die*.

**Imbalance:** class-weighted loss (inverse square-root frequency) plus random 90° rotations and flips, which are valid because a defect pattern keeps its meaning when the wafer is rotated.

**Before running on Colab**
1. *Runtime → Change runtime type → T4 GPU*
2. Colab runs on a temporary Google machine that is wiped when the session ends, so the data has to be sent there. Default (`USE_DRIVE = True`): put `wm811k_labeled_64.npz` in Google Drive at `MyDrive/wafer/`; data and results then survive disconnects. Or set `USE_DRIVE = False` to upload the file each session.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
# True: read data from Google Drive and save results there (survives disconnects).
# False: upload the .npz from your computer each session instead.
USE_DRIVE = True

if IN_COLAB:
    !git clone -q https://github.com/join8112/wafer-yield-portfolio.git
    %cd wafer-yield-portfolio
    !pip install -q -e .
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA = Path('/content/drive/MyDrive/wafer/wm811k_labeled_64.npz')
        OUT = '/content/drive/MyDrive/wafer/outputs'
    else:
        from google.colab import files
        DATA = Path('/content/wm811k_labeled_64.npz')
        if not DATA.exists():
            uploaded = files.upload()  # pick data/processed/wm811k_labeled_64.npz
            Path(next(iter(uploaded))).rename(DATA)
        OUT = '/content/outputs'
else:
    DATA = '../data/processed/wm811k_labeled_64.npz'
    OUT = '../reports'

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from wafer import cnn, evaluate
from wafer.data import FAILURE_TYPES
from wafer.eda import WAFER_CMAP
from wafer.preprocess import load_dataset

OUT = Path(OUT); FIG_DIR = OUT / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
ds = load_dataset(DATA)
tr, va, te = (ds['split'] == s for s in ('train', 'val', 'test'))
print({s: int(m.sum()) for s, m in zip(('train', 'val', 'test'), (tr, va, te))})

## 1. Train (~10 minutes on a T4)

In [ ]:
%%time
model, history = cnn.train(
    ds['X'][tr], ds['y'][tr], ds['X'][va], ds['y'][va],
    epochs=15, batch_size=256, lr=2e-3, device=DEVICE,
)
torch.save(model.state_dict(), OUT / 'wafer_cnn.pt')

In [ ]:
h = pd.DataFrame(history).set_index('epoch')
fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(h.index, h['train_loss'], 'b-o', label='train loss')
ax2 = ax1.twinx(); ax2.plot(h.index, h['val_macro_f1'], 'r-o', label='val macro-F1')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss'); ax2.set_ylabel('val macro-F1')
fig.legend(loc='center right'); plt.title('Training curve'); plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_training_curve.png', dpi=150)

## 2. Test-set results

In [ ]:
probs = cnn.predict(model, ds['X'][te], device=DEVICE)
pred = probs.argmax(1)
summary = evaluate.summarize(ds['y'][te], pred, ds['n_fail'][te], name='CNN')
summary

In [ ]:
evaluate.per_class_report(ds['y'][te], pred)

In [ ]:
fig = evaluate.plot_confusion(ds['y'][te], pred, 'CNN (test, row-normalised)')
fig.savefig(FIG_DIR / 'cnn_confusion.png', dpi=150)

## 3. Grad-CAM: what is the model looking at?

For each class, one correctly classified test wafer. Red dies are failures; the heat-map overlay shows the regions that drove the prediction.

In [ ]:
test_idx = np.flatnonzero(te)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, len(FAILURE_TYPES), figsize=(2 * len(FAILURE_TYPES), 4.4))
for k, name in enumerate(FAILURE_TYPES):
    hits = test_idx[(ds['y'][te] == k) & (pred == k)]
    for ax in axes[:, k]: ax.set_xticks([]); ax.set_yticks([])
    axes[0, k].set_title(name, fontsize=9)
    if len(hits) == 0: continue
    i = rng.choice(hits)
    cam, _ = cnn.grad_cam(model, ds['X'][i], target=k)
    axes[0, k].imshow(ds['X'][i], cmap=WAFER_CMAP, vmin=0, vmax=2)
    axes[1, k].imshow(ds['X'][i] > 0, cmap='gray', vmin=0, vmax=1.5)
    axes[1, k].imshow(cam, cmap='jet', alpha=0.5)
axes[0, 0].set_ylabel('wafer map'); axes[1, 0].set_ylabel('Grad-CAM')
plt.tight_layout(); plt.savefig(FIG_DIR / 'cnn_gradcam.png', dpi=150)

## 4. Save results and copy them into the repo

In [ ]:
report = evaluate.per_class_report(ds['y'][te], pred)
results = {
    'model': 'CNN (3 conv blocks, 2-channel 64x64 input)',
    **summary.round(4).to_dict(),
    'recall_per_class': report['recall'].round(4).to_dict(),
    'epochs': len(history),
}
(OUT / 'results_cnn.json').write_text(json.dumps(results, indent=2))
results

In [ ]:
if IN_COLAB:
    # Downloads results_cnn.json, the figures and the model weights as one zip.
    import shutil
    from google.colab import files
    shutil.make_archive('/content/cnn_outputs', 'zip', OUT)
    files.download('/content/cnn_outputs.zip')

Unzip `cnn_outputs.zip` into the repo's `reports/` folder (`results_cnn.json` → `reports/`, PNGs → `reports/figures/`). Then *File → Download → Download .ipynb* and replace `notebooks/04_cnn_colab.ipynb`, so the outputs show on GitHub.

## Findings

_Send the outputs above to Claude to write this section._